# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook trains and evaluates **`latent-play`** on Google Colab Pro (A100 GPU):
1. **Latent Action Model (LAM)**: Unsupervised discrete action inference from raw gameplay video
2. **Action-conditioned JEPA Dynamics Model**: Predictive representations with EMA target network and anti-collapse regularizers
3. **Visual Decoder**: Imaginary rollout generation and video evaluation
4. **Cross-Entropy Method (CEM) Planning**: Receding-horizon goal-directed control in representation space
5. **Label-Fraction Sweep**: Scaling curve of planning performance vs. action supervision fraction (0.1% to 100%)

## 1. Setup & Hardware Verification

In [ ]:
# Check GPU and hardware capabilities
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Install dependencies in Colab
!pip install -q crafter imageio pyyaml tqdm pytest

## 2. Collect Crafter Dataset

In [ ]:
!python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps 300000 --workers 4
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif

## 3. Train Latent Action Model (LAM)

In [ ]:
# Train LAM on action-free video (bf16 FlashAttention on A100)
!python scripts/train_lam.py --config configs/lam_crafter.yaml train.compile=true train.amp=auto

## 4. Train Action-Conditioned JEPA Dynamics Model

In [ ]:
# Train JEPA dynamics conditioned on latent actions from trained LAM
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.amp=auto

## 5. Closed-Loop CEM Planning in Representation Space

In [ ]:
# Evaluate goal-conditioned navigation using receding-horizon CEM in latent space
!python scripts/eval_planning.py --config configs/dyn_crafter.yaml --checkpoint checkpoints/dyn_crafter/dyn_final.pt --num-episodes 20

In [ ]:
# Display trajectory animation
from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")

## 6. Label-Fraction Sweep (Headline Research Figure)

In [ ]:
# Sweep action supervision fraction: 0.1%, 1%, 5%, 10%, 100%
!python scripts/sweep_labels.py --data-root data/crafter --lam-ckpt checkpoints/lam_crafter/lam_final.pt --fractions 0.001 0.01 0.05 0.1 1.0